In [1]:
import random
import pandas as pd

# ==== ĐẶC TRƯNG NỀN ====
residence_area = {
    'high': ["TP Hồ Chí Minh", "Hà Nội", "Hải Phòng", "Quảng Ninh", "Đà Nẵng",
             "Lạng Sơn", "Lào Cai", "Tây Ninh", "Kiên Giang", "Bình Dương",
             "Đồng Nai", "Cần Thơ", "An Giang", "Bà Rịa - Vũng Tàu"],
    'medium': ["Bình Thuận", "Bình Phước", "Bình Định", "Khánh Hòa", "Nghệ An",
               "Thanh Hóa", "Thừa Thiên Huế", "Đắk Lắk", "Đắk Nông", "Gia Lai",
               "Kon Tum", "Quảng Nam", "Quảng Ngãi", "Phú Yên", "Hà Tĩnh",
               "Hải Dương", "Nam Định", "Ninh Bình", "Thái Nguyên", "Vĩnh Phúc",
               "Long An", "Hậu Giang", "Tiền Giang", "Trà Vinh", "Vĩnh Long",
               "Sóc Trăng", "Cà Mau", "Bạc Liêu", "Bến Tre", "Phú Thọ",
               "Hưng Yên", "Thái Bình", "Ninh Thuận", "Hòa Bình", "Yên Bái",
               "Tuyên Quang", "Hà Nam", "Lâm Đồng", "Quảng Bình"],
    'low': ["Bắc Giang", "Bắc Kạn", "Bắc Ninh", "Cao Bằng", "Điện Biên",
            "Hà Giang", "Sơn La", "Lai Châu", "Quảng Trị", "Đồng Tháp"]
}

occupation = {
    'high': ["Cầm đồ", "Kinh doanh nhà hàng karaoke", "Chủ quán bar", "Làm từ thiện",
             "Tiếp viên quán", "Vũ công tự do", "Streamer", "Youtuber", "Tiktoker",
             "Kinh doanh vàng bạc", "Chơi chứng khoán", "Doanh nhân", "Tự doanh",
             "Môi giới bất động sản", "Kinh doanh đa cấp", "Không rõ"],
    'medium': ["Luật sư", "Nhân viên ngân hàng", "Kế toán", "Kiểm toán viên",
               "Chuyên viên tài chính", "Tư vấn bảo hiểm", "Môi giới chứng khoán",
               "Tài xế giao dịch tiền", "Nhà báo", "Nghệ sĩ tự do", "Ca sĩ", "Diễn viên",
               "MC", "Freelancer", "Nhà văn", "Nhiếp ảnh gia", "Thiết kế đồ họa",
               "Chuyên gia SEO", "Quản trị fanpage", "Lái xe công nghệ", "Bán hàng online",
               "Chủ cửa hàng", "Lái taxi", "Chủ quán ăn", "Nhân viên marketing",
               "Cửa hàng trưởng", "Quản lý khách sạn", "Làm thuê thời vụ", "Trình dược viên",
               "Sinh viên", "Thất nghiệp", "Nội trợ"],
    'low': ["Công an", "Bộ đội", "Thẩm phán", "Kiểm sát viên", "Giảng viên đại học",
            "Giáo viên phổ thông", "Gia sư", "Nhà khoa học", "Kỹ sư phần mềm", "Lập trình viên",
            "Kỹ sư xây dựng", "Kỹ sư điện", "Kỹ thuật viên phòng lab", "Chuyên viên CNTT",
            "Phân tích dữ liệu", "Bác sĩ", "Y tá", "Dược sĩ", "Bác sĩ thú y", 
            "Nhân viên chăm sóc sắc đẹp", "Chuyên viên spa", "Chăm sóc người già", "Công nhân",
            "Thợ xây", "Thợ điện", "Thợ nước", "Thợ mộc", "Thợ hàn", "Lái xe tải", "Bốc vác",
            "Bảo vệ", "Nhân viên bảo trì", "Nhân viên bán hàng", "Nhân viên phục vụ", "Lễ tân",
            "Nhân viên thu ngân", "Tư vấn tuyển sinh", "Học sinh"]
}

def get_label(score):
    if score >= 17:
        return 4
    elif score >= 13:
        return 3
    elif score >= 9:
        return 2
    elif score >= 5:
        return 1
    else:
        return 0

def calculate_score(per_violation_score, per_role_score, per_legal, org_violation, org_legal, alias_count, age, area, job):
    score = 0
    reasons = []

    # --- VI PHẠM CÁ NHÂN ---
    if per_legal == "Minh oan":
        reasons.append("Được minh oan (reset vi phạm và vai trò)")
    else:
        if per_violation_score > 0:
            score += per_violation_score
            reasons.append(f"Vi phạm: +{per_violation_score}")
        if per_role_score > 0:
            score += per_role_score
            reasons.append(f"Vai trò: +{per_role_score}")
        if per_legal == "Đã kết án":
            score += 3; reasons.append("Đã kết án (+3)")
        elif per_legal in ["Đang điều tra", "Truy tố"]:
            score += 2; reasons.append("Đang điều tra/truy tố (+2)")
        elif per_legal == "Chưa rõ":
            score += 1; reasons.append("Pháp lý chưa rõ (+1)")

    # --- VI PHẠM TỔ CHỨC ---
    if org_violation:
        if "tài chính" in org_violation or "cấm vận" in org_violation:
            score += 5; reasons.append("Tổ chức bị chế tài mạnh (+5)")
        elif "ngành" in org_violation:
            score += 4; reasons.append("Tổ chức bị hạn chế ngành (+4)")
        elif "thứ cấp" in org_violation:
            score += 3; reasons.append("Tổ chức bị trừng phạt thứ cấp (+3)")

    if org_legal == "Đã kết án":
        score += 3; reasons.append("Tổ chức đã kết án (+3)")
    elif org_legal in ["Đang điều tra", "Truy tố"]:
        score += 2; reasons.append("Tổ chức bị điều tra/truy tố (+2)")

    # --- ALIAS + TUỔI ---
    if alias_count > 3:
        score += 2; reasons.append("Nhiều alias (+2)")
    elif alias_count > 0:
        score += 1; reasons.append("Có alias (+1)")
    if age < 30 or age > 65:
        score += 1; reasons.append("Tuổi rủi ro (+1)")

    # --- ĐIỂM NỀN ---
    if area in residence_area['high']:
        score += 2; reasons.append("Khu vực rủi ro cao (+2)")
    elif area in residence_area['medium']:
        score += 1; reasons.append("Khu vực rủi ro vừa (+1)")

    if job in occupation['high']:
        score += 3; reasons.append("Nghề rủi ro cao (+3)")
    elif job in occupation['medium']:
        score += 1; reasons.append("Nghề rủi ro vừa (+1)")

    return score, reasons

def generate_aml_data_full_violation(n_samples=1000, seed=2025):
    random.seed(seed)
    samples = []
    unique_samples = set()
    area_list = [loc for sub in residence_area.values() for loc in sub]
    job_list = [job for sub in occupation.values() for job in sub]
    violation_type_pool = {
        6: ["Rửa tiền", "Tài trợ khủng bố"],
        4: ["Lừa đảo", "Chiếm đoạt tài sản", "Tham nhũng", "Hối lộ"],
        2: ["Vi phạm hành chính", "Tranh chấp dân sự", "Vi phạm dân sự", "Vi phạm nhỏ"],
        0: [None]
    }
    legal_pool = ["Đã kết án", "Đang điều tra", "Truy tố", "Chưa rõ", "Minh oan", None]
    role_pool = {
        6: ["Chủ mưu", "Cầm đầu", "Tổ chức thực hiện"],
        4: ["Tham gia", "Giúp sức", "Đồng phạm"],
        2: ["Bị nhắc tên", "Liên quan bị động"],
        0: [None]
    }
    org_violation_pool = [
        "Trừng phạt tài chính", "Cấm vận kinh tế",
        "Trừng phạt ngành", "Trừng phạt thứ cấp"
    ]
    org_legal_pool = ["Đã kết án", "Đang điều tra", "Truy tố", None]
    max_violation = 5
    target_distribution = {
        4: int(n_samples * random.uniform(0.04, 0.06)),
        3: int(n_samples * random.uniform(0.08, 0.12)),
        2: int(n_samples * random.uniform(0.15, 0.25)),
        1: int(n_samples * random.uniform(0.25, 0.35)),
    }
    target_distribution[0] = n_samples - sum(target_distribution.values())
    non_null_sample_count = 0
    for _ in range(n_samples * 15):  # tăng số vòng lặp để đảm bảo số lượng khi bị skip vì trùng
        area = random.choice(area_list)
        job = random.choice(job_list)
        alias_count = random.choices([0, 1, 2, 3, 4], weights=[0.5, 0.2, 0.15, 0.1, 0.05])[0]
        age = random.randint(18, 75)
        per_violation_count = random.choices([1, 2, 3, 4, 5], weights=[0.7, 0.18, 0.07, 0.03, 0.02])[0]
        per_violation_scores = random.choices([6, 4, 2, 0], weights=[0.1, 0.15, 0.3, 0.45], k=per_violation_count)
        per_violation_types = []
        used_types = set()
        for score in per_violation_scores:
            candidates = [x for x in violation_type_pool[score] if x not in used_types]
            vtype = random.choice(candidates) if candidates else None
            per_violation_types.append(vtype)
            if vtype: used_types.add(vtype)
        while len(per_violation_types) < max_violation:
            per_violation_types.append(None)
        per_legal_statuses = [
            random.choices(
                legal_pool,
                weights=[0.03, 0.05, 0.04, 0.1, 0.01, 0.77]
            )[0] if vtype is not None else None
            for vtype in per_violation_types
        ]
        per_violation_score = per_violation_scores[0]
        per_violation = per_violation_types[0]
        per_legal = per_legal_statuses[0]
        if per_violation_score > 0:
            per_role_score = random.choices([6, 4, 2], weights=[0.3, 0.4, 0.3])[0]
            per_role = random.choice(role_pool[per_role_score])
        else:
            per_role_score = 0
            per_role = None
        org_violation_count = random.choices([0, 1, 2, 3, 4, 5], weights=[0.55, 0.25, 0.10, 0.06, 0.03, 0.01])[0]
        org_violation_types = []
        used_org_types = set()
        for _ in range(org_violation_count):
            candidates = [x for x in org_violation_pool if x not in used_org_types]
            vtype = random.choice(candidates) if candidates else None
            org_violation_types.append(vtype)
            if vtype: used_org_types.add(vtype)
        while len(org_violation_types) < max_violation:
            org_violation_types.append(None)
        org_legal_statuses = [
            random.choice(org_legal_pool) if vtype is not None else None
            for vtype in org_violation_types
        ]
        org_violation = org_violation_types[0]
        org_legal = org_legal_statuses[0]
        score, reasons = calculate_score(
            per_violation_score, per_role_score, per_legal,
            org_violation, org_legal,
            alias_count, age, area, job
        )
        label = get_label(score)
        if sum(1 for s in samples if s["label"] == label) >= target_distribution[label]:
            continue
        is_non_null = any([
            per_violation_score > 0,
            per_role_score > 0,
            per_legal not in ("", None),
            org_violation not in ("", None),
            org_legal not in ("", None)
        ])
        if not is_non_null and non_null_sample_count < int(n_samples * 0.7):
            continue
        elif is_non_null:
            non_null_sample_count += 1
        row = {
            "residence_area": area,
            "occupation": job,
            "alias_count": alias_count,
            "age": age,
            "per_role": per_role,
            "total_score": score,
            "label": label,
            "risk_reason": "; ".join(reasons)
        }
        # Add per_violation_type_x, per_legal_status_x
        for i in range(max_violation):
            row[f"per_violation_type_{i+1}"] = per_violation_types[i]
            row[f"per_legal_status_{i+1}"] = per_legal_statuses[i]
        # Add org_violation_type_x, org_legal_status_x
        for i in range(max_violation):
            row[f"org_violation_type_{i+1}"] = org_violation_types[i]
            row[f"org_legal_status_{i+1}"] = org_legal_statuses[i]

        # ===== Kiểm tra trùng lặp toàn bộ sample =====
        # Chuyển toàn bộ row thành tuple để kiểm tra
        sample_tuple = tuple(row.values())
        if sample_tuple in unique_samples:
            continue  # Bỏ qua nếu trùng
        unique_samples.add(sample_tuple)

        samples.append(row)
        if len(samples) >= n_samples:
            break
    return pd.DataFrame(samples)


In [2]:
# residence_area_values = sorted([
#     "TP Hồ Chí Minh", "Hà Nội", "Hải Phòng", "Quảng Ninh", "Đà Nẵng",
#              "Lạng Sơn", "Lào Cai", "Tây Ninh", "Kiên Giang", "Bình Dương",
#              "Đồng Nai", "Cần Thơ", "An Giang", "Bà Rịa - Vũng Tàu", "Bình Thuận", "Bình Phước", "Bình Định", "Khánh Hòa", "Nghệ An",
#                "Thanh Hóa", "Thừa Thiên Huế", "Đắk Lắk", "Đắk Nông", "Gia Lai",
#                "Kon Tum", "Quảng Nam", "Quảng Ngãi", "Phú Yên", "Hà Tĩnh",
#                "Hải Dương", "Nam Định", "Ninh Bình", "Thái Nguyên", "Vĩnh Phúc",
#                "Long An", "Hậu Giang", "Tiền Giang", "Trà Vinh", "Vĩnh Long",
#                "Sóc Trăng", "Cà Mau", "Bạc Liêu", "Bến Tre", "Phú Thọ",
#                "Hưng Yên", "Thái Bình", "Ninh Thuận", "Hòa Bình", "Yên Bái",
#                "Tuyên Quang", "Hà Nam", "Lâm Đồng", "Quảng Bình", "Bắc Giang", "Bắc Kạn", "Bắc Ninh", "Cao Bằng", "Điện Biên",
#             "Hà Giang", "Sơn La", "Lai Châu", "Quảng Trị", "Đồng Tháp"
    
# ])
# occupation_values = sorted(set(occupation['high'] + occupation['medium'] + occupation['low'] + ["None"]))
# per_role_values = sorted([
#     "Chủ mưu", "Cầm đầu", "Tổ chức thực hiện", "Tham gia", "Giúp sức", "Đồng phạm",
#     "Bị nhắc tên", "Liên quan bị động", "None"
# ])
# per_violation_type_values = sorted([
#     "Rửa tiền", "Tài trợ khủng bố", "Lừa đảo", "Chiếm đoạt tài sản", "Tham nhũng", "Hối lộ",
#     "Vi phạm hành chính", "Tranh chấp dân sự", "Vi phạm dân sự", "Vi phạm nhỏ", "None"
# ])
# per_legal_status_values = sorted([
#     "Đã kết án", "Đang điều tra", "Truy tố", "Chưa rõ", "Minh oan", "None"
# ])
# org_violation_type_values = sorted([
#     "Trừng phạt tài chính", "Cấm vận kinh tế", "Trừng phạt ngành", "Trừng phạt thứ cấp", "None"
# ])
# org_legal_status_values = sorted([
#     "Đã kết án", "Đang điều tra", "Truy tố", "None"
# ])
# encoder_config = {
#     "residence_area": residence_area_values,
#     "occupation": occupation_values,
#     "per_role": per_role_values,
#     "per_violation_type": per_violation_type_values,
#     "per_legal_status": per_legal_status_values,
#     "org_violation_type": org_violation_type_values,
#     "org_legal_status": org_legal_status_values,
# }


In [3]:
import random
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [4]:
df = generate_aml_data_full_violation(1000, seed=42)

In [5]:
import pandas as pd

pd.set_option('display.max_columns', None)

# pd.set_option('display.max_rows', None)

pd.set_option('display.expand_frame_repr', False)

pd.set_option('display.width', 200)

df.head(50)

,residence_area,occupation,alias_count,age,per_role,total_score,label,risk_reason,per_violation_type_1,per_legal_status_1,per_violation_type_2,per_legal_status_2,per_violation_type_3,per_legal_status_3,per_violation_type_4,per_legal_status_4,per_violation_type_5,per_legal_status_5,org_violation_type_1,org_legal_status_1,org_violation_type_2,org_legal_status_2,org_violation_type_3,org_legal_status_3,org_violation_type_4,org_legal_status_4,org_violation_type_5,org_legal_status_5
0,Trà Vinh,Lái xe công nghệ,2,18,Cầm đầu,15,3,Vi phạm: +4; Vai trò: +6; Pháp lý chưa rõ (+1)...,Tham nhũng,Chưa rõ,Tranh chấp dân sự,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,Bắc Kạn,Trình dược viên,1,69,Đồng phạm,9,2,Vi phạm: +2; Vai trò: +4; Có alias (+1); Tuổi ...,Vi phạm hành chính,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,Cà Mau,Nhân viên bảo trì,3,41,None,9,2,Tổ chức bị hạn chế ngành (+4); Tổ chức đã kết ...,None,None,None,None,None,None,None,None,None,None,Trừng phạt ngành,Đã kết án,None,None,None,None,None,None,None,None
3,Đắk Nông,Ca sĩ,1,62,None,5,1,Tổ chức bị điều tra/truy tố (+2); Có alias (+1...,None,None,None,None,Hối lộ,None,None,None,None,None,Cấm vận kinh tế,Đang điều tra,Trừng phạt thứ cấp,Đã kết án,Trừng phạt ngành,Truy tố,Trừng phạt tài chính,None,None,None
4,Khánh Hòa,Tiktoker,0,54,Tổ chức thực hiện,17,4,Vi phạm: +2; Vai trò: +6; Pháp lý chưa rõ (+1)...,Vi phạm nhỏ,Chưa rõ,None,None,None,None,None,None,None,None,Trừng phạt ngành,None,None,None,None,None,None,None,None,None
5,Điện Biên,Thợ mộc,0,32,Tham gia,15,3,Vi phạm: +4; Vai trò: +4; Tổ chức bị chế tài m...,Chiếm đoạt tài sản,None,Vi phạm nhỏ,None,None,None,None,None,Tham nhũng,None,Trừng phạt tài chính,Truy tố,None,None,None,None,None,None,None,None
6,Tuyên Quang,Lễ tân,0,36,Bị nhắc tên,11,2,Vi phạm: +2; Vai trò: +2; Tổ chức bị hạn chế n...,Vi phạm dân sự,None,None,None,None,None,None,None,None,None,Trừng phạt ngành,Đang điều tra,Trừng phạt thứ cấp,Đang điều tra,None,None,None,None,None,None
7,Tiền Giang,Chơi chứng khoán,0,49,None,6,1,Tổ chức bị điều tra/truy tố (+2); Khu vực rủi ...,None,None,Tranh chấp dân sự,None,None,None,None,None,None,None,Cấm vận kinh tế,Đang điều tra,Trừng phạt ngành,Đang điều tra,Trừng phạt thứ cấp,Truy tố,None,None,None,None
8,Quảng Nam,Học sinh,1,46,Chủ mưu,20,4,Vi phạm: +2; Vai trò: +6; Đã kết án (+3); Tổ c...,Vi phạm hành chính,Đã kết án,Chiếm đoạt tài sản,None,Rửa tiền,Đang điều tra,None,None,None,None,Trừng phạt tài chính,Đang điều tra,Trừng phạt thứ cấp,Truy tố,None,None,None,None,None,None
9,Bến Tre,Phân tích dữ liệu,0,26,None,5,1,Tổ chức đã kết án (+3); Tuổi rủi ro (+1); Khu ...,None,None,None,None,None,None,None,None,None,None,Cấm vận kinh tế,Đã kết án,Trừng phạt tài chính,None,None,None,None,None,None,None


In [6]:
residence_area_values = sorted([
    "TP Hồ Chí Minh", "Hà Nội", "Hải Phòng", "Quảng Ninh", "Đà Nẵng",
             "Lạng Sơn", "Lào Cai", "Tây Ninh", "Kiên Giang", "Bình Dương",
             "Đồng Nai", "Cần Thơ", "An Giang", "Bà Rịa - Vũng Tàu", "Bình Thuận", "Bình Phước", "Bình Định", "Khánh Hòa", "Nghệ An",
               "Thanh Hóa", "Thừa Thiên Huế", "Đắk Lắk", "Đắk Nông", "Gia Lai",
               "Kon Tum", "Quảng Nam", "Quảng Ngãi", "Phú Yên", "Hà Tĩnh",
               "Hải Dương", "Nam Định", "Ninh Bình", "Thái Nguyên", "Vĩnh Phúc",
               "Long An", "Hậu Giang", "Tiền Giang", "Trà Vinh", "Vĩnh Long",
               "Sóc Trăng", "Cà Mau", "Bạc Liêu", "Bến Tre", "Phú Thọ",
               "Hưng Yên", "Thái Bình", "Ninh Thuận", "Hòa Bình", "Yên Bái",
               "Tuyên Quang", "Hà Nam", "Lâm Đồng", "Quảng Bình", "Bắc Giang", "Bắc Kạn", "Bắc Ninh", "Cao Bằng", "Điện Biên",
            "Hà Giang", "Sơn La", "Lai Châu", "Quảng Trị", "Đồng Tháp"
    
])
occupation_values = sorted(set(occupation['high'] + occupation['medium'] + occupation['low'] + ["None"]))
per_role_values = sorted([
    "Chủ mưu", "Cầm đầu", "Tổ chức thực hiện", "Tham gia", "Giúp sức", "Đồng phạm",
    "Bị nhắc tên", "Liên quan bị động", "None"
])
per_violation_type_values = sorted([
    "Rửa tiền", "Tài trợ khủng bố", "Lừa đảo", "Chiếm đoạt tài sản", "Tham nhũng", "Hối lộ",
    "Vi phạm hành chính", "Tranh chấp dân sự", "Vi phạm dân sự", "Vi phạm nhỏ", "None"
])
per_legal_status_values = sorted([
    "Đã kết án", "Đang điều tra", "Truy tố", "Chưa rõ", "Minh oan", "None"
])
org_violation_type_values = sorted([
    "Trừng phạt tài chính", "Cấm vận kinh tế", "Trừng phạt ngành", "Trừng phạt thứ cấp", "None"
])
org_legal_status_values = sorted([
    "Đã kết án", "Đang điều tra", "Truy tố", "None"
])
encoder_config = {
    "residence_area": residence_area_values,
    "occupation": occupation_values,
    "per_role": per_role_values,
    "per_violation_type": per_violation_type_values,
    "per_legal_status": per_legal_status_values,
    "org_violation_type": org_violation_type_values,
    "org_legal_status": org_legal_status_values,
}


In [7]:
categorical_cols = [
    "residence_area", 'occupation', 'per_role',
    'per_violation_type_1', 'per_legal_status_1', 'per_violation_type_2', 'per_legal_status_2',
    'per_violation_type_3', 'per_legal_status_3', 'per_violation_type_4', 'per_legal_status_4',
    'per_violation_type_5', 'per_legal_status_5',
    'org_violation_type_1', 'org_legal_status_1', 'org_violation_type_2', 'org_legal_status_2',
    'org_violation_type_3', 'org_legal_status_3', 'org_violation_type_4', 'org_legal_status_4',
    'org_violation_type_5', 'org_legal_status_5'
]

features = [col for col in df.columns if col not in ['label', 'risk_reason', 'total_score']]
df_encoded = df[features + ['label']].copy()

In [8]:
label_encoders = {}
label_maps = {}
for col in categorical_cols:
    base_col = col
    # Gom các cột *_1, *_2,... thành base col để lấy đúng giá trị cố định
    if "_" in col and col.rsplit("_", 1)[-1].isdigit():
        if "per_violation_type" in col:
            base_col = "per_violation_type"
        elif "per_legal_status" in col:
            base_col = "per_legal_status"
        elif "org_violation_type" in col:
            base_col = "org_violation_type"
        elif "org_legal_status" in col:
            base_col = "org_legal_status"
    valid_values = encoder_config[base_col]
    le = LabelEncoder()
    le.fit(valid_values)
    df_encoded[col] = le.transform(df_encoded[col].astype(str))
    label_encoders[col] = le
    label_maps[col] = dict(zip(le.classes_, le.transform(le.classes_)))

In [9]:
X = df_encoded[features]
y = df_encoded['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

In [10]:
y_train.head()

437    1
704    0
263    1
191    1
952    0
Name: label, dtype: int64

In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Tạo mô hình với tham số mặc định cơ bản
xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    tree_method='hist',     # chuẩn mới cho GPU
    device='cuda'
)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.948     0.869     0.907        84
           1      0.632     0.796     0.705        54
           2      0.600     0.583     0.592        36
           3      0.364     0.250     0.296        16
           4      0.556     0.500     0.526        10

    accuracy                          0.730       200
   macro avg      0.620     0.600     0.605       200
weighted avg      0.734     0.730     0.728       200



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [12:08:03] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [12]:
import joblib
joblib.dump(xgb_model, 'xgb_aml_model.pkl')

['xgb_aml_model.pkl']

In [13]:
# import joblib
# xgb_model = joblib.load('xgb_aml_model.pkl')

In [14]:
# params = {
#     'max_depth': [3, 5, 7, 9],
#     'n_estimators': [100, 150, 200],
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'subsample': [0.7, 0.8, 0.9, 1.0],
#     'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
#     'gamma': [0, 0.5, 1.0, 2.0],
#     'min_child_weight': [1, 3, 5]
# }

# xgb_model = XGBClassifier(
#     objective='multi:softmax',
#     num_class=5,
#     eval_metric='mlogloss',
#     use_label_encoder=False,
#     random_state=42,
#     tree_method='hist',      # dùng "hist" thay vì "gpu_hist"
#     device='cuda'            # chỉ rõ device là CUDA
# )

# grid = GridSearchCV(
#     estimator=xgb_model,
#     param_grid=params,
#     cv=3,
#     scoring='f1_macro',
#     verbose=0,
#     n_jobs=-1
# )
# grid.fit(X_train, y_train)

# print("Best params:", grid.best_params_)
# print("Best score (cv, f1_macro):", grid.best_score_)
# best_model = grid.best_estimator_

# y_pred = best_model.predict(X_test)
# print(classification_report(y_test, y_pred, digits=3))

# Inference

In [15]:
sample = {
    "residence_area": "TP Hồ Chí Minh",
    "occupation": "Doanh nhân",
    "alias_count": 2,
    "age": 45,
    "per_role": "Chủ mưu",
    "per_violation_type_1": "Rửa tiền",
    "per_legal_status_1": "Đã kết án",
    "per_violation_type_2": "Vi phạm hành chính",
    "per_legal_status_2": "Chưa rõ",
    "per_violation_type_3": "None",
    "per_legal_status_3": "None",
    "per_violation_type_4": "None",
    "per_legal_status_4": "None",
    "per_violation_type_5": "None",
    "per_legal_status_5": "None",
    "org_violation_type_1": "Trừng phạt tài chính",
    "org_legal_status_1": "Đã kết án",
    "org_violation_type_2": "None",
    "org_legal_status_2": "None",
    "org_violation_type_3": "None",
    "org_legal_status_3": "None",
    "org_violation_type_4": "None",
    "org_legal_status_4": "None",
    "org_violation_type_5": "None",
    "org_legal_status_5": "None"
}

In [16]:
import numpy as np
import pandas as pd

# Tạo DataFrame 1 dòng từ sample
df_sample = pd.DataFrame([sample])

# Encode các cột categorical theo label_encoders (đã fit từ trước)
for col in categorical_cols:
    le = label_encoders[col]
    # Nếu value không nằm trong classes_ thì sẽ lỗi, nên phải check & fallback "None"
    val = df_sample.at[0, col]
    if val not in le.classes_:
        df_sample.at[0, col] = "None"
    df_sample[col] = le.transform(df_sample[col].astype(str))

# Đảm bảo đúng thứ tự cột như khi train
X_sample = df_sample[features].values

# Nếu train trên GPU thì model vẫn nhận numpy array thường
y_pred_sample = xgb_model.predict(X_sample)
print("Kết quả dự đoán:", int(y_pred_sample[0]))

Kết quả dự đoán: 4
